## 연습 3: 노트북 범위 라이브러리 설치

HealthBridge Analytics에서는 서로 다른 팀이 때때로 동일한 라이브러리의 서로 다른 버전을 필요로 합니다. 변형마다 새 클러스터를 만드는 대신 라이브러리를 **노트북 범위**로 설치할 수 있습니다. 이러한 라이브러리는 현재 노트북 세션에만 적용되며 동일한 컴퓨팅 리소스의 다른 사용자나 노트북에는 영향을 주지 않습니다.

이 연습에서는 `faker` 라이브러리를 노트북 범위로 설치하고 올바르게 작동하는지 확인합니다.

[faker](https://faker.readthedocs.io/)는 테스트, 프로토타이핑, 데이터베이스 시드에 사용할 이름, 주소, 전화번호, 날짜 등과 같은 사실적인 가짜 데이터를 생성하는 Python 라이브러리입니다. 다양한 로캘과 데이터 범주를 지원하므로 실제 개인 정보를 사용하지 않고도 대량의 그럴듯한 합성 데이터를 쉽게 생성할 수 있습니다.

### 작업 3.1 — `faker` 라이브러리를 노트북 범위로 설치

여러분의 팀은 파이프라인 테스트를 위한 합성 환자 레코드를 생성하기 위해 `faker`를 사용합니다. 이 노트북 세션에서만 사용할 수 있도록 노트북 범위로 설치하세요.

> 🤖 **Genie Code 팁:** 아래 프롬프트를 사용하여 시작하세요:
> *"매직 명령을 사용하여 Databricks에서 Python 패키지를 노트북 범위로 설치하려면 어떻게 해야 해?"*

**힌트:** `%pip` 매직 명령을 사용하여 패키지를 설치하세요. 정확한 버전을 고정하는 것(예: `faker==40.8.0`)은 재현성을 위한 좋은 방법입니다.

In [ ]:
%pip install faker==40.8.0

### 작업 3.2 — 설치 확인

실제 파이프라인에서 `faker`를 사용하기 전에 다음을 통해 올바르게 설치되었는지 확인하세요:

1. `faker` 라이브러리에서 `Faker` 클래스를 가져옵니다.
2. `Faker` 인스턴스를 만듭니다.
3. 무작위로 생성된 **전체 이름**과 **생년월일**을 출력합니다.

> 🤖 **Genie Code 팁:**
> *"Faker 라이브러리를 사용하여 무작위 이름과 생년월일을 생성하는 Python 예제를 보여줘."*

In [ ]:
from faker import Faker

fake = Faker()

print("전체 이름     :", fake.name())
print("생년월일      :", fake.date_of_birth(minimum_age=18, maximum_age=85))

## 연습 4: 합성 환자 데이터 생성 및 분석

이제 `faker`가 설치되었으니 활용해 봅시다. 여러분의 팀은 새 데이터 파이프라인을 테스트하기 위한 환자 입원 레코드의 합성 데이터셋이 필요합니다. 데이터는 변환과 집계를 검증할 수 있을 만큼 충분히 사실적이어야 하지만, HIPAA 정책을 준수하기 위해 **실제 환자 정보를 포함해서는 안 됩니다**.

이 연습에서는 합성 데이터셋을 생성하고 기본 분석을 실행합니다.

### 작업 4.1 — 합성 환자 입원 DataFrame 생성

`faker`와 Spark 세션(`spark`)을 사용하여 **100개의 합성 환자 입원 레코드**가 포함된 **Spark DataFrame**을 만드세요. 각 레코드에는 다음이 포함되어야 합니다:

| 열 | 설명 |
|---|---|
| `patient_id` | 정수, 1부터 100까지 |
| `full_name` | 무작위로 생성된 전체 이름 |
| `date_of_birth` | `1940-01-01`과 `2005-12-31` 사이의 무작위 날짜, 문자열로 |
| `admission_date` | `2023-01-01`과 `2025-12-31` 사이의 무작위 날짜, 문자열로 |
| `diagnosis_code` | 다음 중 하나: `I21`, `J18`, `E11`, `K80`, `N39`(무작위 선택) |

결과 DataFrame의 처음 10개 행을 표시하세요.

> 🤖 **Genie Code 팁:**
> *"Faker 라이브러리를 사용하여 Python 딕셔너리 목록을 생성한 다음 해당 목록에서 PySpark DataFrame을 만들려면 어떻게 해야 해?"*
>
> **힌트:** `minimum_age` 및 `maximum_age` 매개변수와 함께 `faker.date_of_birth()`를 사용하거나 `start_date` 및 `end_date`와 함께 `faker.date_between()`을 사용하세요. 진단 코드에는 `random.choice()`를 사용하세요.

In [ ]:
import random
from faker import Faker
from datetime import date

fake = Faker()

diagnosis_codes = ["I21", "J18", "E11", "K80", "N39"]

records = []
for i in range(1, 101):
    records.append({
        "patient_id": i,
        "full_name": fake.name(),
        "date_of_birth": fake.date_between(start_date=date(1940, 1, 1), end_date=date(2005, 12, 31)),
        "admission_date": fake.date_between(start_date=date(2023, 1, 1), end_date=date(2025, 12, 31)),
        "diagnosis_code": random.choice(diagnosis_codes)
    })

df = spark.createDataFrame(records)
display(df.limit(10))

### 작업 4.2 — 진단 코드별 입원 분석

임상 정보학 팀은 입원 데이터에서 어떤 진단 코드가 가장 자주 나타나는지 알고 싶어 합니다. 작업 4.1에서 만든 Spark DataFrame을 사용하여:

1. `diagnosis_code`별 입원 수를 계산합니다.
2. 입원 수가 **많은 것부터 적은 것** 순으로 결과를 정렬합니다.
3. 결과를 표시합니다.

> 🤖 **Genie Code 팁:**
> *"PySpark DataFrame API를 사용하여 열을 기준으로 그룹화하고, 발생 횟수를 세고, 결과를 내림차순으로 정렬하려면 어떻게 해야 해?"*

In [ ]:
from pyspark.sql.functions import count, desc

admission_counts = (
    df
    .groupBy("diagnosis_code")
    .agg(count("*").alias("admission_count"))
    .orderBy(desc("admission_count"))
)

display(admission_counts)